## 26 — Citing Papers Data Collection

For each award-winning paper we fetch the papers that **cite it** (forward citations).
This is the reverse direction of notebook 24 (`referenced_works` = backward citations).

**OpenAlex mechanism:**  
`GET /works?filter=cites:W<id>&per-page=200&cursor=*`  
This returns all works that cite a given paper, paginated via cursor.

**Workflow:**
1. Load `huang_matched_openalex.csv` → unique award paper OpenAlex IDs
2. For each award paper: paginate through all citing works, collect their IDs
3. Build edge table: `award_paper_id → citing_paper_id`
4. Deduplicate citing IDs, batch-fetch full metadata (same `extract_paper_attrs` as nb-24)
5. Save `citing_papers.csv` and `award_to_citing_edges.csv`

**Output files:**
- `citing_papers.csv` — one row per unique citing paper, same attributes as award papers
- `award_to_citing_edges.csv` — explicit mapping: which award paper is cited by which paper

**Space / performance notes:**
- `counts_by_year` stored as compact JSON string (same as nb-24)
- `citing_award_papers` stored as `|`-separated ID string (not a list)
- Checkpoint every 25 award papers (citing lists can be large → save often)
- Batch size 200 for metadata fetch (max OpenAlex allows)

In [ ]:
import pandas as pd
import requests
import json
import time
from pathlib import Path

MAILTO   = 'shaheryar.4822@student.uu.se'
API_KEY  = 'A08hCjeUoeVKA9toVsfCpF'
BASE_URL = 'https://api.openalex.org'

DATA = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched')

MATCHED_PATH      = DATA / 'huang_matched_openalex.csv'
OUT_CITING        = DATA / 'citing_papers.csv'
OUT_EDGES         = DATA / 'award_to_citing_edges.csv'
CHECKPOINT_EDGES  = DATA / 'citing_edges_checkpoint.csv'
CHECKPOINT_PAPERS = DATA / 'citing_papers_checkpoint.csv'

# Select fields — identical to nb-24 to keep datasets consistent
SELECT_FIELDS = (
    'id,doi,title,publication_year,publication_date,type,'
    'cited_by_count,is_retracted,open_access,primary_location,'
    'best_oa_location,authorships,topics,counts_by_year'
)

def api_get(url, params=None):
    p = {'mailto': MAILTO, 'api_key': API_KEY}
    if params:
        p.update(params)
    for attempt in range(4):
        try:
            r = requests.get(url, params=p, timeout=20)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                time.sleep(10 * (attempt + 1))
        except requests.RequestException:
            time.sleep(3)
    return None

print('Ready.')

### Step 1 — Load award papers

In [ ]:
df = pd.read_csv(MATCHED_PATH)
df = df[df['year'].between(2000, 2018)].copy()
df = df.dropna(subset=['openalex_id'])

award_papers = (
    df.drop_duplicates(subset='openalex_id')
    [['openalex_id', 'year', 'conference', 'paper_title']]
    .copy()
)
award_papers['openalex_id'] = award_papers['openalex_id'].str.strip()

print(f'Total award papers (2000-2018): {len(award_papers)}')
print(f'Year range: {award_papers["year"].min()} – {award_papers["year"].max()}')
print(award_papers.head(3))

### Step 2 — Paginate citing works for each award paper

OpenAlex cursor pagination: start with `cursor=*`, then follow `meta.next_cursor` until `None`.
We only store IDs at this stage to keep memory low.

In [ ]:
def fetch_citing_ids(award_oa_id):
    """Return list of (citing_paper_id,) for all papers citing award_oa_id."""
    short_id = award_oa_id.split('/')[-1]
    cursor   = '*'
    ids      = []

    while cursor:
        data = api_get(
            f'{BASE_URL}/works',
            params={
                'filter'  : f'cites:{short_id}',
                'select'  : 'id',
                'per-page': 200,
                'cursor'  : cursor,
            }
        )
        time.sleep(0.12)

        if not data:
            break

        results = data.get('results', [])
        ids.extend(r['id'] for r in results if r.get('id'))

        cursor = data.get('meta', {}).get('next_cursor')  # None when last page

    return ids


# Resume from checkpoint if available
if CHECKPOINT_EDGES.exists():
    edges_df = pd.read_csv(CHECKPOINT_EDGES)
    done_ids = set(edges_df['award_paper_id'].unique())
    edges    = edges_df.to_dict('records')
    print(f'Resuming — {len(done_ids)} award papers already processed, {len(edges):,} edges')
else:
    edges    = []
    done_ids = set()

to_process = award_papers[~award_papers['openalex_id'].isin(done_ids)]
print(f'Award papers left to process: {len(to_process)}')

for i, row in enumerate(to_process.itertuples(), 1):
    oa_id   = row.openalex_id
    cit_ids = fetch_citing_ids(oa_id)

    for cid in cit_ids:
        edges.append({
            'award_paper_id'   : oa_id,
            'award_year'       : row.year,
            'award_conference' : row.conference,
            'citing_paper_id'  : cid,
        })

    if i % 25 == 0:
        pd.DataFrame(edges).to_csv(CHECKPOINT_EDGES, index=False)
        print(f'  [{i}/{len(to_process)}] checkpoint — {len(edges):,} edges | last paper: {len(cit_ids)} citers')

edges_df = pd.DataFrame(edges)
edges_df.to_csv(CHECKPOINT_EDGES, index=False)

print(f'\nTotal citing edges      : {len(edges_df):,}')
print(f'Unique citing paper IDs : {edges_df["citing_paper_id"].nunique():,}')

### Step 3 — Build `cites_n_award_papers` lookup & deduplicate

In [ ]:
# For each citing paper: list of award papers it cites + count
cites_map = (
    edges_df.groupby('citing_paper_id')['award_paper_id']
    .apply(lambda x: '|'.join(sorted(x.unique())))
    .reset_index()
    .rename(columns={'award_paper_id': 'cites_award_papers'})
)

cites_count = (
    edges_df.groupby('citing_paper_id')['award_paper_id']
    .nunique()
    .reset_index()
    .rename(columns={'award_paper_id': 'cites_n_award_papers'})
)

citing_meta     = cites_map.merge(cites_count, on='citing_paper_id')
unique_cite_ids = citing_meta['citing_paper_id'].tolist()

print(f'Unique citing papers to enrich : {len(unique_cite_ids):,}')
print(f'Cite >= 2 award papers          : {(citing_meta["cites_n_award_papers"] >= 2).sum():,}')
print(f'Cite >= 5 award papers          : {(citing_meta["cites_n_award_papers"] >= 5).sum():,}')

### Step 4 — Batch-fetch citing paper metadata

Batch size 200 (OpenAlex OR filter limit).  
Same `extract_paper_attrs` as notebook 24 for dataset consistency.

In [ ]:
def extract_paper_attrs(w):
    primary_loc = w.get('primary_location') or {}
    source      = primary_loc.get('source') or {}

    authorships      = w.get('authorships', [])
    author_ids       = '|'.join([a['author']['id'] for a in authorships if a.get('author') and a['author'].get('id')])
    author_names     = '|'.join([a['author'].get('display_name', '') for a in authorships if a.get('author')])
    author_positions = '|'.join([a.get('author_position', '') for a in authorships])

    first_insts = []
    if authorships:
        first_insts = [i.get('display_name', '') for i in authorships[0].get('institutions', [])]

    topics    = w.get('topics', [])
    top_topic = topics[0].get('display_name', '') if topics else ''
    top_field = topics[0].get('field', {}).get('display_name', '') if topics else ''

    counts_by_year = w.get('counts_by_year', [])

    return {
        'openalex_id'              : w.get('id', ''),
        'doi'                      : w.get('doi', ''),
        'title'                    : w.get('title', ''),
        'publication_year'         : w.get('publication_year'),
        'publication_date'         : w.get('publication_date', ''),
        'type'                     : w.get('type', ''),
        'cited_by_count'           : w.get('cited_by_count', 0),
        'is_retracted'             : w.get('is_retracted', False),
        'is_oa'                    : w.get('open_access', {}).get('is_oa', False),
        'source_id'                : source.get('id', ''),
        'source_name'              : source.get('display_name', ''),
        'source_type'              : source.get('type', ''),
        'source_issn'              : '|'.join(source.get('issn', []) or []),
        'author_ids'               : author_ids,
        'author_names'             : author_names,
        'author_positions'         : author_positions,
        'author_count'             : len(authorships),
        'first_author_institution' : '|'.join(first_insts),
        'top_topic'                : top_topic,
        'top_field'                : top_field,
        'counts_by_year'           : json.dumps(counts_by_year),
    }

In [ ]:
def fetch_works_batch(id_list):
    """Fetch up to 200 works in one request using OR filter."""
    ids_str = '|'.join([i.split('/')[-1] for i in id_list])
    data = api_get(
        f'{BASE_URL}/works',
        params={
            'filter'  : f'openalex_id:{ids_str}',
            'per-page': 200,
            'select'  : SELECT_FIELDS,
        }
    )
    if data:
        return data.get('results', [])
    return []


# Resume from checkpoint
if CHECKPOINT_PAPERS.exists():
    existing     = pd.read_csv(CHECKPOINT_PAPERS)
    fetched_ids  = set(existing['openalex_id'].tolist())
    all_papers   = existing.to_dict('records')
    print(f'Resuming — {len(fetched_ids):,} citing papers already fetched')
else:
    fetched_ids = set()
    all_papers  = []

remaining = [i for i in unique_cite_ids if i not in fetched_ids]
print(f'Citing papers left to fetch: {len(remaining):,}')

BATCH   = 200
batches = [remaining[i:i+BATCH] for i in range(0, len(remaining), BATCH)]

for b_idx, batch in enumerate(batches, 1):
    results = fetch_works_batch(batch)
    time.sleep(0.15)

    for w in results:
        all_papers.append(extract_paper_attrs(w))

    if b_idx % 50 == 0:
        pd.DataFrame(all_papers).to_csv(CHECKPOINT_PAPERS, index=False)
        pct = b_idx / len(batches) * 100
        print(f'  [batch {b_idx}/{len(batches)} | {pct:.1f}%] {len(all_papers):,} papers fetched')

citing_papers_df = pd.DataFrame(all_papers)
citing_papers_df.to_csv(CHECKPOINT_PAPERS, index=False)
print(f'\nTotal citing papers fetched: {len(citing_papers_df):,}')

### Step 5 — Merge metadata & save final outputs

In [ ]:
final_citing = citing_papers_df.merge(
    citing_meta, left_on='openalex_id', right_on='citing_paper_id', how='left'
)
final_citing = final_citing.drop(columns=['citing_paper_id'], errors='ignore')

# Compact dtypes to save disk space
for col in ['cited_by_count', 'author_count', 'cites_n_award_papers']:
    if col in final_citing.columns:
        final_citing[col] = pd.to_numeric(final_citing[col], errors='coerce').astype('Int32')

final_citing['publication_year'] = pd.to_numeric(
    final_citing['publication_year'], errors='coerce'
).astype('Int16')

# Save
final_citing.to_csv(OUT_CITING, index=False)
edges_df.to_csv(OUT_EDGES, index=False)

print('=' * 60)
print(f'citing_papers.csv         → {len(final_citing):,} rows')
print(f'award_to_citing_edges.csv → {len(edges_df):,} rows')
print(f'\ncites_n_award_papers distribution (top 10):')
print(final_citing['cites_n_award_papers'].value_counts().sort_index().head(10))
print(f'\nMissing cites_award_papers: {final_citing["cites_award_papers"].isna().sum()}')
print(f'\nYear range of citing papers:')
print(f'  min: {final_citing["publication_year"].min()}')
print(f'  max: {final_citing["publication_year"].max()}')
print(f'  median: {final_citing["publication_year"].median()}')
print(f'\nSample rows:')
print(
    final_citing[['openalex_id', 'title', 'publication_year',
                   'cited_by_count', 'cites_n_award_papers']]
    .head(3).to_string()
)